In [1]:
import requests
import pandas as pd

api_key = "f59da65fa2c1487a84c8d61e0c76ad61"
base_url = "http://api.football-data.org/v4"

headers = {
    "X-Auth-Token": api_key
}

#Teste de conexão:
response = requests.get(f"{base_url}/competitions", headers=headers)

print(f"Status: {response.status_code}")
print(f"Conexão: {'Tudo certo!' if response.status_code == 200 else 'Algo errado'}")

Status: 200
Conexão: Tudo certo!


In [3]:
#Buscando dados:

response = requests.get(f"{base_url}/competitions/BSA/matches", headers=headers)

print(f"Status: {response.status_code}")
data = response.json()
print(data)

Status: 200
{'filters': {'season': '2026'}, 'resultSet': {'count': 380, 'first': '2026-01-28', 'last': '2026-12-02', 'played': 63}, 'competition': {'id': 2013, 'name': 'Campeonato Brasileiro Série A', 'code': 'BSA', 'type': 'LEAGUE', 'emblem': 'https://crests.football-data.org/bsa.png'}, 'matches': [{'area': {'id': 2032, 'name': 'Brazil', 'code': 'BRA', 'flag': 'https://crests.football-data.org/764.svg'}, 'competition': {'id': 2013, 'name': 'Campeonato Brasileiro Série A', 'code': 'BSA', 'type': 'LEAGUE', 'emblem': 'https://crests.football-data.org/bsa.png'}, 'season': {'id': 2474, 'startDate': '2026-01-28', 'endDate': '2026-12-02', 'currentMatchday': 7, 'winner': None}, 'id': 554740, 'utcDate': '2026-01-28T22:00:00Z', 'status': 'FINISHED', 'matchday': 1, 'stage': 'REGULAR_SEASON', 'group': None, 'lastUpdated': '2026-03-18T20:21:04Z', 'homeTeam': {'id': 1766, 'name': 'CA Mineiro', 'shortName': 'Mineiro', 'tla': 'CAM', 'crest': 'https://crests.football-data.org/1766.png'}, 'awayTeam': {

In [4]:
#Extraindo do json:
matches = data['matches']

#Transformando em dataframe:
df = pd.DataFrame([{
    'data':        m['utcDate'],
    'rodada':      m['matchday'],
    'status':      m['status'],
    'time_casa':   m['homeTeam']['name'],
    'time_fora':   m['awayTeam']['name'],
    'gols_casa':   m['score']['fullTime']['home'],
    'gols_fora':   m['score']['fullTime']['away'],
} for m in matches])

#Ajuste de data:
df['data'] = pd.to_datetime(df['data']).dt.date

#Conferência:
print(f"Total de jogos coletados: {len(df)}")
print(f"Rodadas: {df['rodada'].min()} a {df['rodada'].max()}")
df.head(10)

Total de jogos coletados: 380
Rodadas: 1 a 38


,data,rodada,status,time_casa,time_fora,gols_casa,gols_fora
0,2026-01-28,1,FINISHED,CA Mineiro,SE Palmeiras,2.0,2.0
1,2026-01-28,1,FINISHED,Coritiba FBC,RB Bragantino,0.0,1.0
2,2026-01-28,1,FINISHED,SC Internacional,CA Paranaense,0.0,1.0
3,2026-01-28,1,FINISHED,EC Vitória,Clube do Remo,2.0,0.0
4,2026-01-28,1,FINISHED,Fluminense FC,Grêmio FBPA,2.0,1.0
5,2026-01-28,1,FINISHED,Chapecoense AF,Santos FC,4.0,2.0
6,2026-01-28,1,FINISHED,SC Corinthians Paulista,EC Bahia,1.0,2.0
7,2026-01-29,1,FINISHED,São Paulo FC,CR Flamengo,2.0,1.0
8,2026-01-29,1,FINISHED,Mirassol FC,CR Vasco da Gama,2.0,1.0
9,2026-01-30,1,FINISHED,Botafogo FR,Cruzeiro EC,4.0,0.0


In [5]:
#Verificando dados completos:

print(f"Período dos jogos: {df['data'].min()} até {df['data'].max()}")
print(f"\nStatus dos jogos:")
print(df['status'].value_counts())
print(f"\nTotal de jogos FINALIZADOS: {len(df[df['status'] == 'FINISHED'])}")

Período dos jogos: 2026-01-28 até 2026-12-02

Status dos jogos:
status
SCHEDULED    300
FINISHED      63
TIMED         14
POSTPONED      3
Name: count, dtype: int64

Total de jogos FINALIZADOS: 63


In [6]:
#Filtrando apenas jogos finalizados:
df_finalizado = df[df['status'] == 'FINISHED'].copy()

#Resetando index:
df_finalizado = df_finalizado.reset_index(drop=True)

#Verificando:
print(f"Jogos finalizados: {len(df_finalizado)}")
print(f"Rodadas disputadas: {df_finalizado['rodada'].min()} a {df_finalizado['rodada'].max()}")
print(f"\nPrimeiros jogos:")
df_finalizado.head(10)

Jogos finalizados: 63
Rodadas disputadas: 1 a 7

Primeiros jogos:


,data,rodada,status,time_casa,time_fora,gols_casa,gols_fora
0,2026-01-28,1,FINISHED,CA Mineiro,SE Palmeiras,2.0,2.0
1,2026-01-28,1,FINISHED,Coritiba FBC,RB Bragantino,0.0,1.0
2,2026-01-28,1,FINISHED,SC Internacional,CA Paranaense,0.0,1.0
3,2026-01-28,1,FINISHED,EC Vitória,Clube do Remo,2.0,0.0
4,2026-01-28,1,FINISHED,Fluminense FC,Grêmio FBPA,2.0,1.0
5,2026-01-28,1,FINISHED,Chapecoense AF,Santos FC,4.0,2.0
6,2026-01-28,1,FINISHED,SC Corinthians Paulista,EC Bahia,1.0,2.0
7,2026-01-29,1,FINISHED,São Paulo FC,CR Flamengo,2.0,1.0
8,2026-01-29,1,FINISHED,Mirassol FC,CR Vasco da Gama,2.0,1.0
9,2026-01-30,1,FINISHED,Botafogo FR,Cruzeiro EC,4.0,0.0


In [9]:
#Salvando dados brutos:

df_finalizado.to_csv('brasileirao_2026.csv', index=False)

print("Dados salvos em 'brasileirao_2026.csv'")
print(f"\nResumo geral:")
print(f"Jogos coletados: {len(df_finalizado)}")
print(f"Rodadas: {df_finalizado['rodada'].min()} a {df_finalizado['rodada'].max()}")
print(f"Times únicos: {pd.concat([df_finalizado['time_casa'], df_finalizado['time_fora']]).nunique()}")

Dados salvos em 'brasileirao_2026.csv'

Resumo geral:
Jogos coletados: 63
Rodadas: 1 a 7
Times únicos: 20
